# train2: temporary training notebook (single-teacher control of the main line: instructed 8K-embed)

The main result, `main_v3dom_s21600` (1k-base student, BGE-M3 + instructed 8K-embed, mix v3 + domain texts, 450K, lr 1e-4, 21,600 steps), got TR-MTEB **68.27** (seed 43: 68.30). The claim "two teachers beat the best single teacher" has so far only been shown at 5400 steps on the 200K mix (pair 67.16 vs BGE 66.07 and 8K 65.67, +1.09). Single students gained a lot from more budget before, so the gap must be measured at the main-line recipe.

This notebook trains `single_v3dom_instr8k_s21600`: the same recipe (`configs/mix_v3dom_single_instr8k.yaml`), only the instructed 8K-embed teacher (questions with the card's query instruction, the other texts from the plain cache). The BGE-M3 single teacher is in train1; the two can run at the same time on separate runtimes.

Both caches already exist (the main-line run used them): nothing is encoded. Approximate A100 time: ~3 h (0.44 s/step; a single-teacher step is a little cheaper). Re-running the cells after a disconnect resumes the cut run; the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook. (name, config, extra overrides). A later override wins.
BASE = []
RUNS = [
    ('single_v3dom_instr8k_s21600', 'configs/mix_v3dom_single_instr8k.yaml', []),   # single-teacher control, reference main_v3dom_s21600 (68.27 / seed 43 68.30)
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()